# Script to add technical replicates to long table and prepare mouse m2 specific library

## 0. Pre-adjusted settings
### 0.1. Packages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
from Bio.Seq import Seq
from itertools import product
import inspect
import matplotlib as plt


### 0.2. Paths

In [ ]:
#---------------------------------
#---------------------------------
#---------------------------------

#general Path

general_dir = Path('/Users/kollybook/Library/Mobile Documents/com~apple~CloudDocs/Kolja_Hildenbrand/Uni/Master_Infectious_Diseases/Grimm_internship/Report_Grimm/Data') # <----- Hier den Server Path anpassen
os.makedirs(general_dir, exist_ok=True)

#---------------------------------
#---------------------------------
#---------------------------------

# Path for tech_rep
NGS_tech_rep_dir = general_dir/'NGS_data'/'tech_rep'
os.makedirs(NGS_tech_rep_dir, exist_ok=True)

# Path for tables
tables_dir = general_dir/'tables'
os.makedirs(tables_dir, exist_ok=True)




### 0.3 Definition of functions for processing the tables

In [ ]:
def ext_sample_folders(folder_dir):
    folders = sorted([
        f.name 
        for f in folder_dir.iterdir() 
        if not f.name.startswith(".")
    ])
    return folders

#### 0.3.2. Create lists based on sample folders

In [ ]:
def extract_lists (folders):
    tissues = []
    ext_types = []

    for folder in folders:
        parts = folder.split("_")

        ext, tissue = parts

        if ext not in ext_types:
            ext_types.append(ext)

        if tissue not in tissues:
            tissues.append(tissue)

    return tissues, ext_types

#### 0.3.3. Load csv files

In [ ]:
def load_csv_files(path, folders, mouse_ids):
    """
    Load CSV files into nested dict:
    NGS_files[tissue][ext][mouse_id] = dataframe
    """

    path = Path(path)
    NGS_files = {}

    for folder in folders:
        folder_path = path / folder

        parts = folder.split("_")

        ext = parts[0]     # cDNA or gDNA
        tissue = parts[1]  # liver or heart

        NGS_files.setdefault(tissue, {}).setdefault(ext, {})

        # ext-specific prefix
        if ext == "cDNA":
            prefix = "c"
        elif ext == "gDNA":
            prefix = "g"

        # alle csv files im Ordner
        csv_files = [f for f in folder_path.iterdir() if f.is_file() and f.suffix.lower() == ".csv"]

        for mouse_id in mouse_ids:
            matching_files = []

            for file in csv_files:
                name_lower = file.name.lower()

                if name_lower.startswith(prefix) and mouse_id.lower() in name_lower:
                    matching_files.append(file)
                    
            df = pd.read_csv(matching_files[0])
            NGS_files[tissue][ext][mouse_id] = df

    return NGS_files
            

#### 0.3.4. Load input librarys for tissue and extraction_type

In [ ]:
# Load input library
def load_input_library ():
    df = pd.read_csv(NGS_input_dir / 'input_lib_outputg35_input_lib_S35_R1_001_PV.csv')
    input_library = {}
    for tissue in TISSUES:
        input_library[tissue] = {}
        for ext in EXT:
            input_library[tissue][ext] = df.copy()
    return input_library

#### 0.3.5. translate nt --> AA sequence

In [ ]:
def translate_to_AA(files):
    AA_files = {}

    for tissue in TISSUES:
        AA_files[tissue] = {}

        for ext in EXT:
            item = files[tissue][ext]

            # check if file input_library 
            if isinstance(item, pd.DataFrame):
                df = item.copy()
                df["AA_sequence"] = df['Peptide'].apply(lambda x: str(Seq(x).translate()))

                df_aa = (
                    df.groupby("AA_sequence", as_index=False)['Count']
                    .sum()
                    .sort_values('Count', ascending=False)
                    .reset_index(drop=True)
                )

                AA_files[tissue][ext] = df_aa

            # Check if file is sample with Mouse_ID
            elif isinstance(item, dict):
                AA_files[tissue][ext] = {}

                for sample_key, df in item.items():
                    df = df.copy()
                    df["AA_sequence"] = df['Peptide'].apply(lambda x: str(Seq(x).translate()))

                    df_aa = (
                        df.groupby("AA_sequence", as_index=False)['Count']
                        .sum()
                        .sort_values('Count', ascending=False)
                        .reset_index(drop=True)
                    )

                    AA_files[tissue][ext][sample_key] = df_aa

    return AA_files


#### 0.3.6. Remove AA_seq with stop codon

In [ ]:
def remove_stop_codon (files):
    AA_no_stop_files = {}

    for tissue in TISSUES:
        AA_no_stop_files[tissue]= {}

        for ext in EXT:
            item = files[tissue][ext]

            # check if file input_library 
            if isinstance(item, pd.DataFrame):
                df = item.copy()
                df = df[~df["AA_sequence"].str.contains(r"\*")]
                
                AA_no_stop_files[tissue][ext] = df
            # check if dict (Samples)
            elif isinstance(item, dict):
                AA_no_stop_files[tissue][ext] = {}
                
                for sample_key, df in item.items():
                    df = df.copy()
                    df = df[~df["AA_sequence"].str.contains(r"\*")]
                    
                    AA_no_stop_files[tissue][ext][sample_key] = df
                
    return AA_no_stop_files

#### 0.3.7. Merge AA_seq together RENAME

In [ ]:
### OLD
def create_long_AA_table (files):
    long_table = {}

    for tissue in TISSUES:
        long_table[tissue]= {}

        for ext in EXT:
            long_table[tissue][ext]= {}
            long = []
            for sample in file_names[ext]:
                df = files[tissue][ext][sample].copy()
                long.append(df)
    
            df = pd.concat(long, ignore_index=True)
    
            df = df['AA_sequence'].unique()

            long_table[tissue][ext] = df
    return long_table

In [ ]:
# merge all mouse ID together and create pseudo AA_seq and frameshift +1 for sample and input_library
def create_tissue_AA_string(files):
    AA_tissue_string = {}

    for tissue in TISSUES:
        all_aa = []

        for ext in EXT:
            names = list(files[tissue][ext].keys())
            for sample in names:
                df = files[tissue][ext][sample].copy()
                all_aa.extend(df["AA_sequence"].tolist())

        # unique AA sequences pro Tissue
        AA_tissue_string[tissue] = pd.Series(all_aa).drop_duplicates().tolist()

    return AA_tissue_string

#### 0.3.7. Create Pseudo library and frame shift Merge will function before

In [ ]:
def expand_input_library_with_sample_AA(long_table, input_lib):
    expanded_input = {}

    for tissue in TISSUES:
        expanded_input[tissue] = {}

        for ext in EXT:
            aa_sample = long_table[tissue]

            #input library
            df_input = input_lib[tissue][ext].copy()

            # Outer merge mit allen AA_seq aus Samples
            df_merged = pd.DataFrame({"AA_sequence": aa_sample}).merge(
                df_input,
                on="AA_sequence",
                how="outer"
            )
            
            # wenn Count fehlt -> neu durch Samples hinzugefügt
            df_merged["Pseudo"] = 0
            mask = df_merged["Count"].isna()
            df_merged.loc[mask, "Pseudo"] = 1
            
            df_merged["Count"] = df_merged["Count"].fillna(0)
            
            df_merged = (
                df_merged
                .sort_values("Count", ascending=False)
                .reset_index(drop=True)
            )
            df_merged['Count'] = df_merged['Count']+1
            expanded_input[tissue][ext] = df_merged
            

    return expanded_input

#### 0.3.8. Create Pseudo samples with frameshift

In [ ]:
def merge_samples_with_input(files, input_lib):
    merged_files = {}

    for tissue in TISSUES:
        merged_files[tissue] = {}

        for ext in EXT:
            merged_files[tissue][ext] = {}

            # input library for tissue and ext
            df_input = input_lib[tissue][ext].copy()
            df_input = df_input[['AA_sequence', 'Count', 'Pseudo', 'Proportion']]

            # rename input columns
            rename_dict = {
                "Count": "Count_input",
                "Pseudo": "Pseudo_input",
                "Proportion": "Proportion_input"
            }
            df_input = df_input.rename(columns=rename_dict)

            names = list(files[tissue][ext].keys())
            for sample in names:
                df_sample = files[tissue][ext][sample].copy()

                # merge sample + input
                df_merged = df_sample.merge(
                    df_input,
                    on="AA_sequence",
                    how="outer"
                )

                # NaN = 0
                df_merged["Pseudo"] = 0
                mask = df_merged["Count"].isna()
                df_merged.loc[mask, "Pseudo"] = 1
                df_merged = df_merged.fillna(0)

                global_factor = df_merged["Count"].sum() / df_merged["Count_input"].sum()
                df_merged['Count'] = df_merged['Count'] + global_factor

                merged_files[tissue][ext][sample] = df_merged

    return merged_files

#### 0.3.9. Calculate proportion

In [ ]:
def calculate_tables (files):
    AA_calc_files = {}

    for tissue in TISSUES:
        AA_calc_files[tissue] = {}

        for ext in EXT:
            item = files[tissue][ext]

            # if input library
            if isinstance(item, pd.DataFrame):
                df = item.copy()

                total_count = df["Count"].sum()
                df["Proportion"] = df["Count"] / total_count
                df["Cum_prop"] = df["Proportion"].cumsum()

                df = df.reset_index(drop=True)

                AA_calc_files[tissue][ext] = df

            # if sample with mouse_ID
            elif isinstance(item, dict):
                AA_calc_files[tissue][ext] = {}

                for sample_key, df in item.items():
                    df = df.copy()

                    total_count = df["Count"].sum()

                    total_count = df["Count"].sum()
                    df["Proportion"] = df["Count"] / total_count
                    df["Cum_prop"] = df["Proportion"].cumsum()
                    df['Log2_enrichment'] = np.log2(df['Proportion']/df['Proportion_input'])

                    AA_calc_files[tissue][ext][sample_key] = df[['AA_sequence', 'Count', 'Proportion', 'Cum_prop', 'Log2_enrichment', 'Pseudo_input', 'Pseudo']].sort_values("Log2_enrichment", ascending=False).reset_index(drop=True)

    return AA_calc_files

#### 0.3.10. Create long table with all samples

In [ ]:
def extract_prefix(name):
    parts = name.split("_")

    first = parts[0]
    second = parts[1]
    third = parts[2]

    if first.startswith("g"):
        ext = "gDNA"
    elif first.startswith("c"):
        ext = "cDNA"

    return f"{ext}_{second}_{third}"

In [ ]:
def create_long_table (tables):

    value_columns = ['Count', 'Proportion', 'Cum_prop', 'Log2_enrichment', 'Pseudo_input', 'Pseudo']

    # Create new columns for sample identification 
    df_long = []
    for tissue, ext in product(TISSUES, EXT):
        for sample in file_names[ext]:
            df = tables[tissue][ext][sample].copy()
            sample_name = extract_prefix(sample)
            
            df["Sample"] = f'{sample_name}_rep'
            df["Sex"] = 'male'
            df["Mouse_ID"] = 'm2'
            df["Tissue"] = tissue
            df["Extraction_type"] = ext
            df_long.append(df)
    df_long_table_metadata = pd.concat(df_long, ignore_index=True)

    #Reorganization of columns, first AA_sequence, 2nd Sample type, 3rd values
    df_long_table_metadata = df_long_table_metadata[
        [c for c in df_long_table_metadata.columns if c not in value_columns] + value_columns
    ].copy()
    df_long_table = df_long_table_metadata[['AA_sequence', 'Sample']+ value_columns].copy()
    return df_long_table_metadata, df_long_table

### 0.4. Definition of helper functions
#### 0.4.1. Save tables

In [ ]:
# Save tables
def save_tables (path, table, name):
    os.makedirs(path, exist_ok=True)
    table.to_csv(path / f"{name}.csv", index=False)
    

## 1. Script

### 1.1. Create lists

In [ ]:
FOLDER_NAMES = ext_sample_folders (NGS_tech_rep_dir)
TISSUES, EXT = extract_lists(FOLDER_NAMES)
MOUSE_ID = ['f1', 'f2', 'f3', 'm1', 'm2', 'm3']


In [ ]:
FOLDER_NAMES, TISSUES, EXT

### 1.2. Load pre-made tables

In [ ]:
# load liver specific library
dict_AA_input_no_stop = {}
dict_AA_input_no_stop['liver'] = {}
for ext in EXT:
    df = pd.read_csv(tables_dir/'liver'/'df_liver_input_library.csv')[['AA_sequence', 'Count']]
    df['Count'] =  df['Count']-1
    dict_AA_input_no_stop['liver'][ext] = df

# load long table 
df_long_table = pd.read_csv(tables_dir/'summary'/'df_long_all_samples_metadata.csv')
    

### 1.3. Load, translate and remove AA_seq with stop_codon from samples

In [ ]:
# load NGS data of technical replicates

# =========================
# 1) BUILD rep_folder LIST
# =========================
rep_folder = []
for ext in EXT:
    for tis in TISSUES:
        rep_folder.append(f"{ext}_technical_rep_{tis}")

print("rep_folder:", rep_folder)

# =========================
# 2) READ ALL CSVs FROM INPUT FOLDERS
#    input_root/<rep>/*.csv
# =========================
NGS_files = {}
NGS_files['liver']= {}

for ext in EXT:
    NGS_files['liver'][ext] = {}

for ext in EXT:
    for tis in TISSUES:
        rep = f"{ext}_{tis}"
        in_dir = NGS_tech_rep_dir / rep

        csv_files = sorted(in_dir.glob("*.csv"))

        target_dict = NGS_files['liver']['gDNA'] if ext == "gDNA" else NGS_files['liver']['cDNA']

        for f in csv_files:
            # key = filename without suffix
            key = f.stem
            target_dict[key] = pd.read_csv(f)

        print(f"{ext} {tis}: loaded {len(csv_files)} CSV files from {in_dir}")

# =========================
# 4) QUICK SANITY PRINT
# =========================
print("\nLoaded tables:")
print(f"gDNA_rep: {len(NGS_files['liver']['gDNA'])} tables")
print(f"cDNA_rep: {len(NGS_files['liver']['cDNA'])} tables")

# show keys
print("\nExample keys:")
print("gDNA:", list(NGS_files['liver']['gDNA'].keys())[:5])
print("cDNA:", list(NGS_files['liver']['cDNA'].keys())[:5])

In [ ]:
file_names = {}
for ext in EXT:
    file_names[ext] = list(NGS_files['liver'][ext].keys())
    
file_names

In [ ]:
%%time
# translate nt files to AA files
dict_AA_samples = translate_to_AA (NGS_files)

# remove AA_seq with stop codons
dict_AA_samples_no_stop = remove_stop_codon (dict_AA_samples)

#### 1.3.1. Save raw technical replicates tables

In [ ]:
for ext in EXT:
    for sample in file_names[ext]:
            file_name = extract_prefix(sample)
            save_tables(tables_dir/'technical_rep', dict_AA_samples_no_stop['liver'][ext][sample], f'df_{file_name}_raw')

### 1.4. Create tissue and gDNA/cDNA specific input library with frameshift +1

In [ ]:
# merge all mouse ID together and create pseudo AA_seq and frameshift +1 for sample and input_library
unique_AA_seq = create_tissue_AA_string (dict_AA_samples_no_stop)
dict_sample_input = expand_input_library_with_sample_AA (unique_AA_seq, dict_AA_input_no_stop)

# calculate prop and cum_prop for input
AA_input_calc = calculate_tables (dict_sample_input)

### 1.5. calculate proportion and log2_enrichment for samples with frameshift + (sum(count_sample)/sum(count_input))

In [ ]:
# Merge samples with calc_input
dict_sample_merged = merge_samples_with_input (dict_AA_samples_no_stop, AA_input_calc)

# calc proportion, cumulative prop and Log2_enrichment for samples
dict_samples = calculate_tables (dict_sample_merged)

In [ ]:
dict_samples

### 1.6. Create long table of technical replicates and combine with biological replicates

In [ ]:
%%time
# create long table with metadata for ploting and one condense to save
df_long_table_metadata_rep, df_long_table_rep = create_long_table(dict_samples)

In [ ]:
df_long_table_metadata_rep

In [ ]:
# Add column 'rep' to indicate if biological or technical replicate
df_long_table_metadata_rep['Replicate'] ='technical'
df_long_table['Replicate'] ='biological'

In [ ]:
# Combine biological replicate long table with technical rep long table
df_long_table_combined = pd.concat(
    [df_long_table, df_long_table_metadata_rep],
    ignore_index=True
)

df_long_table_combined

### 1.7. Save technical replicates and long table with biological and technical replicates

In [ ]:
# save long table with bio replicates and technical replicates
save_tables(tables_dir/'summary', df_long_table_combined, 'df_long_combined_biological_technical_rep')

In [ ]:
#save all technical replicate calculated samples
for ext in EXT:
    for sample in file_names[ext]:
        file_name = extract_prefix(sample)
        save_tables(tables_dir/'technical_rep', dict_samples['liver'][ext][sample], f'df_{file_name}_m2_processed')

# DONE